# Apache Parquet - Python

All 7 Python examples from [docs/parquet.md](https://platob.github.io/yggdryl/parquet/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
import pathlib
import tempfile

import pyarrow as pa
import pyarrow.parquet as pq

from yggdryl import IOBase

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string()),
])
batch = pa.record_batch(
    {"id": [1, 2, 3], "symbol": ["AAPL", None, "MSFT"]}, schema=schema
)

# The name says Parquet, so no call names an encoding.
path = pathlib.Path(tempfile.mkdtemp()) / "trades.parquet"
with IOBase(path) as handle:
    handle.write_arrow_batch_reader(batch)

    # Reading streams: one batch at a time, never one materialized table.
    assert handle.read_arrow_batch_reader().read_all().num_rows == 3

# The scope published the file at its exact length, so PyArrow's own reader
# finds the footer where the format says it is.
assert pq.read_table(path) == pa.Table.from_batches([batch])

## Column pushdown

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

stored = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string(), nullable=False),
    pa.field("price", pa.float64(), nullable=False),
    pa.field("venue", pa.string(), nullable=False),
])
rows = 4_096
batch = pa.record_batch(
    {
        "id": list(range(rows)),
        "symbol": ["AAPL"] * rows,
        "price": [1.5] * rows,
        "venue": ["XNAS"] * rows,
    },
    schema=stored,
)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.parquet")
handle.write_arrow_batch_reader(batch)

# Two of the four columns, declared as this read's schema.
options = handle.record_options()
options.schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("price", pa.float64(), nullable=False),
])

projected = handle.read_arrow_batch_reader(options=options).read_all()
assert projected.column_names == ["id", "price"]

# Less is read, and the bytes say so rather than the clock.
whole = handle.read_arrow_batch_reader().read_all()
assert projected.nbytes * 2 <= whole.nbytes

# The file is unchanged: it still stores all four.
assert len(handle.read_arrow_field().data_type) == 4

## Options

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
batch = pa.record_batch({"id": list(range(1_000))}, schema=schema)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.parquet")

# Parquet's own settings and the shared ones are properties of one value.
options = handle.record_options()
options.max_row_group_size = 4_096
options.key_value_metadata = {"iceberg.schema-id": "7"}
options.batch_size = 256
options.root_name = "trade"

assert options.max_row_group_size == 4_096
assert options.key_value_metadata == {"iceberg.schema-id": "7"}
assert options.batch_size == 256
assert options.root_name == "trade"
assert not options.safe

handle.write_arrow_batch_reader(batch, options=options)

# batch_size bounds the reader, so no batch holds all 1,000 rows.
counts = [part.num_rows for part in handle.read_arrow_batch_reader(options=options)]
assert sum(counts) == 1_000
assert all(count <= 256 for count in counts), counts

# The root name names the Field recovered from the footer.
assert handle.read_arrow_field(options=options).name == "trade"

## Compression

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

root = pathlib.Path(tempfile.mkdtemp())
rows = 4_000
schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False),
    pa.field("symbol", pa.string()),
])
table = pa.table(
    {"id": list(range(rows)), "symbol": ["AAPL"] * rows}, schema=schema
)

sizes = []
for compression in ("uncompressed", "snappy", "zstd(1)"):
    handle = IOBase(root / f"trades-{compression}.parquet")
    # One batch per read, so the comparison is not split by the default bound.
    options = handle.record_options()
    options.compression = compression
    options.batch_size = rows
    handle.write_arrow_batch_reader(table, options=options)

    # Nothing on the read side names the compression: the footer records it.
    read = handle.read_arrow_batch_reader(options=options).read_all()
    assert read.num_rows == rows, compression
    sizes.append(handle.size)

assert sizes[0] > sizes[1] and sizes[0] > sizes[2], sizes

## Coded handles are rejected

In [ ]:
import pathlib
import tempfile

import pyarrow as pa
import pytest

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])

# The name declares gzip over the Parquet file.
handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.parquet.gz")

with pytest.raises(ValueError, match="parquet compresses"):
    handle.write_arrow_batch_reader(pa.record_batch({"id": [1]}, schema=schema))

# Nothing was published.
assert handle.size == 0

## Field identifiers

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([
    pa.field("id", pa.int64(), nullable=False, metadata={"PARQUET:field_id": "1"}),
    pa.field("symbol", pa.string(), metadata={"PARQUET:field_id": "2"}),
])
batch = pa.record_batch({"id": [1], "symbol": ["AAPL"]}, schema=schema)

handle = IOBase(pathlib.Path(tempfile.mkdtemp()) / "trades.parquet")
handle.write_arrow_batch_reader(batch)

# The ids went into the file, so the recovered Field answers by id rather
# than by position.
recovered = handle.read_arrow_field()
assert [child.parquet_field_id for child in recovered.data_type] == [1, 2]

## The handle underneath

In [ ]:
import pathlib
import tempfile

import pyarrow as pa

from yggdryl import IOBase

schema = pa.schema([pa.field("id", pa.int64(), nullable=False)])
root = pathlib.Path(tempfile.mkdtemp())

# Nothing has been written, so there is nothing to read.
empty = IOBase(root / "absent.parquet")
assert empty.read_arrow_batch_reader().read_all().num_rows == 0

# An empty write still publishes a readable file with the schema in its footer.
handle = IOBase(root / "written.parquet")
handle.write_arrow_batch_reader(pa.Table.from_batches([], schema=schema))
assert handle.size > 0
assert handle.read_arrow_batch_reader().read_all().num_rows == 0
assert len(handle.read_arrow_field().data_type) == 1